### Exploration du dataset et idées pour le modèle de prédiction

In [2]:
#Charger les libraries nécessaires
import sys
import os
import duckdb
# Add the project root to the Python path
sys.path.append(os.path.abspath('..'))

import pandas as pd
from utilities import customdataloader

Bloc réutilisable pour le loading année par année

In [3]:
# Ajuster ce chemin selon votre setup Google Drive Desktop
DATA_ROOT = r"G:\.shortcut-targets-by-id\1SR7TxhNQjFyS8f2bbuo9pcLvzNMLBMwY\CSD\data"

In [4]:
# Instancier le loader
loader = customdataloader(
    pricefile=os.path.join(DATA_ROOT, 'prices', 'prices.parquet'),
    costfile=os.path.join(DATA_ROOT, 'costs', 'costs.parquet'),
    dsimfile=[os.path.join(DATA_ROOT, 'sim_daily', f'sim_daily_{y}.parquet') for y in [2020,2021,2022,2023]],
    msimfile=[os.path.join(DATA_ROOT, 'sim_monthly', f'sim_monthly_{y}.parquet') for y in [2020,2021,2022,2023]],
)

In [5]:
# Chargement complet de costs + prices (ces fichiers sont petits)
costs = pd.read_parquet(os.path.join(DATA_ROOT, 'costs', 'costs.parquet'))
prices = pd.read_parquet(os.path.join(DATA_ROOT, 'prices', 'prices.parquet'))

print(f"costs:  {costs.shape[0]:,} rows  |  prices: {prices.shape[0]:,} rows")

costs:  9,092 rows  |  prices: 453,167 rows


In [6]:
# Explorer un mois spécifique
# Choisir le mois cible et le cutoff correspondant
# Choisir le mois cible et le cutoff correspondant
TARGET_MONTH = '2020-02'       # M+1 : le mois qu'on veut prédire
CUTOFF_DATE  = '2020-01-07'    # 7 de M : moment de décision
 
# Trouver le bon fichier sim_monthly selon l'année cible
target_year = TARGET_MONTH[:4]
sim_file = os.path.join(DATA_ROOT, 'sim_monthly', f'sim_monthly_{target_year}.parquet')
 
# Construire filterdf à partir des EIDs présents dans les simulations du mois cible
sim_eids = duckdb.query(f"""
    SELECT DISTINCT EID
    FROM read_parquet('{sim_file}')
    WHERE strftime(DATETIME, '%Y-%m') = '{TARGET_MONTH}'
""").to_df()
 
known_eids = sim_eids['EID'].values
filterdf = pd.DataFrame({
    'EID': list(known_eids) * 2,
    'MONTH': [TARGET_MONTH] * len(known_eids) * 2,
    'PEAKID': [0] * len(known_eids) + [1] * len(known_eids),
})
print(f"Candidats pour {TARGET_MONTH}: {len(filterdf)} triplets ({len(known_eids)} EIDs uniques)")

Candidats pour 2020-02: 9076 triplets (4538 EIDs uniques)


In [7]:
filterdf.head(5) # Afficher les 5 premiers triplets candidats

,EID,MONTH,PEAKID
0,530,2020-02,0
1,568,2020-02,0
2,580,2020-02,0
3,584,2020-02,0
4,592,2020-02,0


In [8]:
#Simulations mensuelles (pour le mois cible)
monthly_sim = loader.get_monthly_data(filterdf)
print(f"Sim monthly: {monthly_sim.shape}")
monthly_sim.head()

Sim monthly: (9475344, 13)


,SCENARIOID,EID,DATETIME,PEAKID,m_ACTIVATIONLEVEL,m_WINDIMPACT,m_SOLARIMPACT,m_HYDROIMPACT,m_NONRENEWBALIMPACT,m_EXTERNALIMPACT,m_LOADIMPACT,m_TRANSMISSIONOUTAGEIMPACT,m_PSM
0,1,1125,2020-02-01 00:00:00,0,1.634400,-5.353300,0.0,-11.1511,19.6954,-4.8988,-0.9039,3.4335,0.0
1,1,1125,2020-02-01 01:00:00,0,9.476100,13.089800,0.0,11.8470,-17.6423,4.4433,0.5405,-1.4922,0.0
2,1,1125,2020-02-01 02:00:00,0,15.587900,18.202400,0.0,11.4899,-15.2728,3.9731,-0.0317,-1.5181,0.0
3,1,1125,2020-02-01 03:00:00,0,29.146700,27.025299,0.0,11.3364,-10.1010,5.1073,-1.3958,-1.5370,0.0
4,1,1125,2020-02-01 04:00:00,0,33.768299,34.120300,0.0,11.2845,-12.7205,4.9159,-1.0266,-1.3589,0.0


In [11]:
# filterdf pour daily sims = mois M (décision), pas M+1 (cible)
daily_filterdf = pd.DataFrame({
    'EID': list(known_eids) * 2,
    'MONTH': [CUTOFF_DATE[:7]] * len(known_eids) * 2,  # '2020-01' au lieu de '2020-02'
    'PEAKID': [0] * len(known_eids) + [1] * len(known_eids),
})

len(daily_filterdf)

9076

In [13]:
#Simulations journalières (jusqu'au cutoff)
daily_sim = loader.get_daily_data(daily_filterdf, cutoff_date=CUTOFF_DATE)
print(f"Sim daily: {daily_sim.shape}")
daily_sim.head()

Sim daily: (2287152, 13)


,SCENARIOID,EID,DATETIME,PEAKID,d_ACTIVATIONLEVEL,d_WINDIMPACT,d_SOLARIMPACT,d_HYDROIMPACT,d_NONRENEWBALIMPACT,d_EXTERNALIMPACT,d_LOADIMPACT,d_TRANSMISSIONOUTAGEIMPACT,d_PSD
0,1,2,2020-01-01 01:00:00,0,38.148499,12.6173,0.0,5.9827,13.1583,1.6088,1.9061,-5.9611,0.0
1,1,2,2020-01-01 02:00:00,0,36.768700,13.5673,0.0,5.8844,9.6836,2.3517,2.4012,-5.9489,0.0
2,1,2,2020-01-01 03:00:00,0,35.448700,12.4558,0.0,5.8438,9.0488,2.2406,2.9768,-5.8125,0.0
3,1,2,2020-01-01 04:00:00,0,35.409698,13.6630,0.0,5.8284,8.3701,2.0937,2.5709,-5.8844,0.0
4,1,2,2020-01-01 05:00:00,0,36.243500,14.0931,0.0,6.0008,8.9773,2.1207,2.1685,-5.9449,0.0


In [17]:
(daily_filterdf["EID"]==filterdf["EID"]).value_counts()

EID
True    9076
Name: count, dtype: int64

### Comprendre la structure de base du data

In [20]:
# Combien d'EIDs, heures, scénarios
print(monthly_sim.groupby('SCENARIOID')['EID'].nunique())
print(f"Heures par EID/scénario: {monthly_sim.groupby(['SCENARIOID','EID']).size().describe()}")
print(f"Répartition PEAKID:\n{monthly_sim['PEAKID'].value_counts()}")

SCENARIOID
1    4538
2    4538
3    4538
Name: EID, dtype: int64
Heures par EID/scénario: count    13614.0
mean       696.0
std          0.0
min        696.0
25%        696.0
50%        696.0
75%        696.0
max        696.0
dtype: float64
Répartition PEAKID:
PEAKID
0    5118864
1    4356480
Name: count, dtype: int64


Structure des données monthly_sim

- **3 scénarios** de simulation (SCENARIOID 1, 2, 3), chacun contenant exactement **4 538 EIDs**.
- Chaque combinaison (scénario, EID) contient **696 heures** → correspond aux 29 jours × 24h de février 2020 (année bissextile).
- **Répartition ON/OFF-peak** : plus de lignes OFF-peak (5 118 864) que ON-peak (4 356 480), ce qui reflète le fait que les heures hors-pointe (nuits, weekends) sont plus nombreuses.
- **Total** : 9 475 344 lignes = 3 × 4 538 × 696
- Les `filterdf` et `daily_filterdf` partagent les mêmes EIDs mais visent des mois différents : `filterdf` cible M+1 (simulations mensuelles futures) et `daily_filterdf` cible M (observations journalières passées jusqu'au cutoff).